### Baseline model for preeclampsia

In [ ]:
import sys
import pandas as pd

sys.path.append('../')
sys.path.append('../../src')

df = pd.read_parquet('../preeclampsia_all_features.parquet')
df.shape

In [ ]:
df_pivoted = df.pivot(index='birthid', columns='feature_name', values='feature_value').reset_index()
df_pivoted.columns.name = ''
df_pivoted

In [ ]:
df_pivoted['chtn_any'].isna().sum()

In [ ]:
from preprocess import preprocess

df_pivoted_preprocessed = preprocess(df_pivoted)

In [ ]:
df_pivoted_preprocessed['target_variable'].mean()

In [ ]:
df_pivoted_preprocessed

In [ ]:
from model_training import nested_cv_evaluation_feature_selection
from sklearn.ensemble import RandomForestClassifier

X = df_pivoted_preprocessed.drop(columns=['target_variable', 'birthid'])
print(X.shape)
y = df_pivoted_preprocessed['target_variable']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [30, 50, 70, 100, 200, 300, 500, 700, 1000]}

metrics_baseline = nested_cv_evaluation_feature_selection(X, y, clf, param_grid,
                                                          outer_splits=5, inner_splits=5, random_state=range(10),
                                                          return_for_curves=True)

In [ ]:
from my_scorers import boostrap_ci_agg

metrics_baseline[['roc_auc', 'ap', 'sens_80']].apply(boostrap_ci_agg)

In [ ]:
from my_scorers import plot_nested_cv_with_95ci

fig = plot_nested_cv_with_95ci(metrics_baseline)

In [ ]:
metrics_baseline['best_params']

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = df_pivoted_preprocessed.drop(columns=['target_variable', 'birthid'])
print(X.shape)
y = df_pivoted_preprocessed['target_variable']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', min_samples_split=50)
clf.fit(X, y)

In [ ]:
import shap

def get_shap(clf, X, max_display=30):
    X_sample = X.sample(n=1000, replace=False)
    explainer = shap.TreeExplainer(clf, data=X_sample,
                                   model_output='probability', feature_perturbation='interventional')
    shap_values = explainer(X_sample, check_additivity=True)
    shap_values.values = shap_values.values[:,:,1]
    shap.plots.beeswarm(shap_values, max_display=max_display)
    return shap_values

shap_values = get_shap(clf, X, max_display=50)